In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

  Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (12 kB)
  Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (62 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
  Using cached stevedore-5.7.0-py3-none-any.whl.metadata (2.4 kB)
  Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.2 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
Using cached qiskit-1.2.4-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (4.8 MB)
Using cached symengine-0.13.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (49.7 MB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)
Using cached scipy-1.17.1-cp311-cp311-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (35.3 MB)
Using cached stevedore-5.7.0-py3-none-any.whl (54 kB)
Using cached sympy-1.14.0-py3-none-any.whl

In [2]:
from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile 
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate
import math 

# The aim of the assignment is to simulate the Ekert91 key distribution protocol.

# This notebook is for a simulation of the protocol with an attacker, to demonstrate that the attacker can be detected.

In [10]:
# Quantum Key Distribution Protocol E91 with Attacker
S_values = []

# Constants
root2 = math.sqrt(2)
root3 = math.sqrt(3)

# Key length
N = 20

# Number of times to run
runs = math.ceil((9*N)/2)

# Denominators 
denom1 = math.sqrt(4 + 2*root2)
denom2 = math.sqrt(4 - 2*root2) 

# Alice Operators

# A1 = X
A1_matrix = [[1/root2, 1/root2],
             [1/root2, -1/root2]]
A1 = Operator(A1_matrix)

# A2 = W
A2_matrix = [[-1/denom1, (1 + root2)/denom1],
             [1/denom2, (root2 - 1)/denom2]]
A2 = Operator(A2_matrix)

# A3 = Z
A3_matrix = [[1, 0],
             [0, 1]]
A3 = Operator(A3_matrix)

alice_operators = [A1, A2, A3]


# Bob Operators 

# B1 = W
B1_matrix = [[-1/denom1, (1 + root2)/denom1],
             [1/denom2, (root2 - 1)/denom2]]
B1 = Operator(B1_matrix)

# B2 = Z
B2_matrix = [[1, 0],
             [0, 1]]
B2 = Operator(B2_matrix)

# B3 = V
B3_matrix = [[1/denom1, (1 + root2)/denom1],
             [-1/denom2, (root2 - 1)/denom2]]
B3 = Operator(B3_matrix)

bob_operators = [B1, B2, B3]


# Function Random with probability 1/3
# used to choose one of the three operators for Alice and Bob
# put qubit in state 1/√3|0⟩ + √2/√3|1⟩
# if result is 0 select first operator
# if result is 1 then use another probability 1/2 between the 2nd and 3rd operator 

def random_operator():
    one_third_matrix = [[1/root3, -root2/root3],
                        [root2/root3, 1/root3]]
    q1 = QuantumCircuit(1) 
    q1.unitary(one_third_matrix, [0])
    q1.measure_all() 
    backend = BasicSimulator()
    compiled = transpile(q1, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    chosen = counts.get("1",0)
    if chosen == 0:
        return 0
    q2 = QuantumCircuit(1) 
    q2.h(0)
    q2.measure_all() 
    backend = BasicSimulator()
    compiled = transpile(q2, backend)
    job_sim = backend.run(compiled, shots=1)
    result_sim = job_sim.result() 
    counts = result_sim.get_counts(compiled)
    chosen = counts.get("1",0)
    if chosen == 0:
        return 1
    return 2


# Function Entangled Pairs using state 1/√2(|01⟩ − |10⟩)
def entangledPair():
    q = QuantumCircuit(2) 
    # entangle qubits gives state 1/√2(|00⟩ + |11⟩)
    q.h(0)
    q.cx(0,1)
    # move to correct state 1/√2(|01⟩ − |10⟩)
    q.x(1)
    q.z(1)
    return q


# Function Converter maps {0,1} to {+1,-1}
def converter(bit):
   return 1 if bit == 0 else -1


# Function Average returns the average mean of a list
def average(l):
    if len(l) == 0:
       return 0
    return sum(l)/len(l)


def protocol(attack=None):
    # Storage lists for Alice and Bob measurement choices and results
    alice_choices = []
    alice_results = []
    
    bob_choices = []
    bob_results = []
    
    shared_key = []
    
    bell_alice_bits = []
    bell_alice_choices = []
    
    bell_bob_bits = []
    bell_bob_choices = []
    
    # Protocol 
    # Shared key length N
    # run 9N/2 times
    # execute the 5 steps 
    
    # step 1 Alice and Bob each receive one qubit of an entangled pair in state 1/√2(|01⟩ − |10⟩).
    # step 2 Alice chooses an operator Ai randomly from her set of three, with probability  1/3 each.
    # step 3 Bob chooses an operator Bj randomly from his set of three, with probability 1/3 each.
    # step 4 Alice measures operator Ai on her qubit.
    # step 5 Bob measures operator Bj on his qubit
    
    # measurement results and operator choices are stored 
    
    for i in range(runs):
    
        # step 1 
        q = entangledPair()

        # attack
        q = attack(q)
        
        # step 2 
        alice_choice = random_operator()
        alice_choices.append(alice_choice)
    
        # step 3 
        bob_choice = random_operator()
        bob_choices.append(bob_choice)
    
        # step 4 
        q.unitary(alice_operators[alice_choice], [0])
    
        # step 5 
        q.unitary(bob_operators[bob_choice], [1])
    
        q.measure_all()
    
        # simulate
        backend = BasicSimulator()
        compiled = transpile(q, backend)
        job_sim = backend.run(compiled, shots=1)
        result_sim = job_sim.result() 
        counts = result_sim.get_counts(compiled)
    
        # results
        state = list(counts.keys())[0]
        alice_result = int(state[1])
        bob_result = int(state[0])
    
        alice_results.append(alice_result)
        bob_results.append(bob_result)
    
    
    # Key Generation 
    
    # Alice and Bob reveal sequence of operator choices, i and j for each repetition 
    
    # In following cases:
    # i = 2 and j = 1  (shifted down 1)
    # i = 3 and j = 2  (shifted down 1)
    # the results should be the opposite (same basis) and form part of the final key
    
    # Bob inverts his bits
    
    for alice_choice, bob_choice, alice_result, bob_result in zip(alice_choices, bob_choices, alice_results, bob_results):
    
        if (alice_choice == 1 and bob_choice == 0) or (alice_choice == 2 and bob_choice == 1):
            if len(shared_key) < N:
                bob_bit_inverted = 1 - bob_result
                if alice_result == bob_bit_inverted:
                    shared_key.append(alice_result)
        else: 
           bell_alice_bits.append(alice_result)
           bell_bob_bits.append(bob_result)
           bell_alice_choices.append(alice_choice)
           bell_bob_choices.append(bob_choice)
    
    
    # Security Test (Bell Inequality)
    
    # Remaining results are revealed and used to calculate |S|, The Bell Inequality Test
    # S = |⟨X ⊗ W⟩ − ⟨X ⊗ V⟩ + ⟨Z ⊗ W⟩ + ⟨Z ⊗ V⟩|
    
    # Each expectation value is calculated by:
    # 1. Converting measurement results from {0,1} to {+1,-1}
    # 2. Multiplying Alice's and Bob's values
    # 3. Averaging over many measurements
    
    # if result S ≈ 2√2 then they are entangled
    # if result S <= 2, the attacker has interfered with the qubits and the correlations become classical
    
    
    XW = []
    XV = []
    ZW = []
    ZV = []
    
    for alice_choice, bob_choice, alice_bit, bob_bit in zip(bell_alice_choices, bell_bob_choices, bell_alice_bits, bell_bob_bits):
        # step 1 
        alice_value = converter(alice_bit)
        bob_value = converter(bob_bit)
    
        # step 2 
        # S = |⟨X ⊗ W⟩ − ⟨X ⊗ V⟩ + ⟨Z ⊗ W⟩ + ⟨Z ⊗ V⟩|
        # S = |⟨A1 ⊗ B1⟩ − ⟨A1 ⊗ B3⟩ + ⟨A3 ⊗ B1⟩ + ⟨A3 ⊗ B3⟩|   the operators
        # S = |⟨0 ⊗ 0⟩ − ⟨0 ⊗ 2⟩ + ⟨2 ⊗ 0⟩ + ⟨2 ⊗ 2⟩|           the value of the operators in the lists given from random_operator
        if alice_choice == 0 and bob_choice == 0:
           XW.append(alice_value*bob_value)
        elif alice_choice == 0 and bob_choice == 2:
           XV.append(alice_value*bob_value)
        elif alice_choice == 2 and bob_choice == 0:
           ZW.append(alice_value*bob_value)
        elif alice_choice == 2 and bob_choice == 2:
           ZV.append(alice_value*bob_value)
    
    # step 3 
    S = abs(average(XW) - average(XV) + average(ZW) + average(ZV))
    S_values.append(S)
    
    # Results
    # print out shared key
    # print out Bell test value S
    # print if the qubits are entangled (secure key)
    # or if correlations indicate possible interception
    print("Shared key (length %i): %s" % (len(shared_key), shared_key))
    print(f"Bell test value S: {S}")
    if S <= 2:
       print("The attacker has interfered with the qubits and the correlations have become classical")
    else:
       print("The qubits are entangled so it is a secure key")
    
    # print([["A1", "A2", "A3"][i] for i in alice_choices])
    # # print(alice_choices)
    # print(alice_results)
    
    # print([["B1", "B2", "B3"][i] for i in bob_choices])
    # # print(bob_choices)
    # print(bob_results)
    
    # print([["A1", "A2", "A3"][i] for i in bell_alice_choices])
    # # print(bell_alice_choices)
    # print(bell_alice_bits)
    
    # print([["B1", "B2", "B3"][i] for i in bell_bob_choices])
    # # print(bell_bob_choices)
    # print(bell_bob_bits)



# Function Attack 1 applying a Pauli X gate 
def attack1(q):
    q.x(1)
    return q
    
# Function Attack 2 applying a H gate
def attack2(q):
    q.h(1)
    return q

# Function Attack 3 resetting the qubits to state |0⟩ destroying entanglement
def attack3(q):
    q.reset(0)
    q.reset(1)
    return q

# Function Attack 4 intercepts and entangles their own qubit
def attack4(q):
    new_q = QuantumCircuit(3)
    new_q.h(0)
    new_q.cx(0,1)
    new_q.x(1)
    new_q.z(1)
    new_q.cx(1,2)
    return new_q
    

for attack in [attack1, attack2, attack3, attack4]:
    print(f"Running {attack.__name__}")
    protocol(attack)
    print("")

print(f" The average S value is: {average(S_values)}")

Running attack1
Shared key (length 6): [0, 0, 0, 1, 1, 1]
Bell test value S: 0.10303030303030314
The attacker has interfered with the qubits and the correlations have become classical

Running attack2
Shared key (length 17): [0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1]
Bell test value S: 0.7740259740259741
The attacker has interfered with the qubits and the correlations have become classical

Running attack3
Shared key (length 3): [1, 1, 0]
Bell test value S: 2.080808080808081
The qubits are entangled so it is a secure key

Running attack4
Shared key (length 9): [0, 0, 0, 0, 0, 0, 1, 0, 0]
Bell test value S: 0.9589743589743589
The attacker has interfered with the qubits and the correlations have become classical

 The average S value is: 0.9792096792096792
